# How sampling works (nano_llm)

This notebook walks through **autoregressive generation**: how the model produces one token at a time, and how **greedy**, **top-k**, **top-p (nucleus)**, **temperature**, and **repetition penalty** change the next-token distribution.

For **meaningful** greedy vs top-p text, run **§10** with a **trained** `best.pt` (or train a few epochs on Tiny Shakespeare). §9 uses random weights only to show the API.

Implementation reference: `src/nano_llm/inference/generate.py`  
CLI: `python scripts/generate.py` (see `--method`, `--top-k`, `--top-p`, `--temperature`, `--repetition-penalty`, `--seed`).

## 0) Setup

Run from the repo root or from `docs/`.

In [1]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = next((p for p in candidates if (p / "src" / "nano_llm").exists()), None)
if repo_root is None:
    raise RuntimeError("Could not locate repo root containing src/nano_llm")
sys.path.insert(0, str(repo_root / "src"))

import torch
import torch.nn.functional as F

# Sampling helpers (same code path as production generate())
from nano_llm.inference.generate import (
    _apply_repetition_penalty,
    _greedy_sample,
    _top_k_sample,
    _top_p_sample,
    generate,
    sanitize_output,
)

## 1) Autoregressive loop (big picture)

1. Start from a **prompt**; encode it to token ids `ids`.
2. Repeat up to `max_new_tokens` times:
   - Take the last `max_context` ids (sliding window), forward through the model.
   - Read **last-position logits** `logits` (one score per vocabulary token).
   - Optionally: **temperature** (scale logits), **repetition penalty** (tweak logits for tokens already in `ids`).
   - Pick **one** next id: greedy argmax, or sample from **top-k** / **top-p** distribution.
   - Append id; stop early if newline or `stop_sequence` appears in decoded text.
3. Decode `ids` back to a string; optionally `sanitize_output`.

So “sampling” here means **how you turn logits into the next integer token**.

## 2) From logits to probabilities: softmax and temperature

For last-position logits vector `logits[v]`:

- **Softmax** turns scores into a probability mass over the vocabulary.
- **Temperature** $T$: use `logits / T` before softmax.
  - $T < 1$ → sharper distribution (more peaked on high logits).
  - $T > 1$ → flatter distribution (more randomness).
  - $T = 1$ → no change.

Greedy decoding skips sampling: it takes `argmax(logits)` (temperature still affects argmax if applied before argmax, as in this codebase).

In [2]:
logits = torch.tensor([2.0, 1.0, 0.0, -1.0])
for T in (0.5, 1.0, 2.0):
    p = F.softmax(logits / T, dim=-1)
    print(f"T={T}: probs = {p.tolist()}")

T=0.5: probs = [0.8649548292160034, 0.1170589029788971, 0.015842201188206673, 0.0021440088748931885]
T=1.0: probs = [0.6439142823219299, 0.23688283562660217, 0.08714432269334793, 0.032058604061603546]
T=2.0: probs = [0.45505422353744507, 0.2760043442249298, 0.16740509867668152, 0.101536326110363]


## 3) Greedy sampling

Always pick the token with the **largest logit**. Deterministic (no randomness unless ties).

`_greedy_sample` returns `logits.argmax(-1).item()`.

In [3]:
logits = torch.tensor([0.1, 5.0, 3.0, 2.0])
print("greedy id:", _greedy_sample(logits))

greedy id: 1


## 4) Top-k sampling

1. Keep only the **k largest** logits.
2. **Softmax only on those k** values → distribution over k tokens.
3. **`torch.multinomial`** draws one index; map back to vocabulary id.

If `k <= 0` or `k >= vocab`, the code **falls back to greedy**.

Effect: you **never** sample from the long tail outside the top-k bucket.

In [4]:
torch.manual_seed(0)
logits = torch.tensor([1.0, 3.0, 2.0, 0.5, -2.0])
k = 2
top_k_logits, top_k_idx = torch.topk(logits, k, dim=-1)
probs = F.softmax(top_k_logits.float(), dim=-1)
print("top-k indices:", top_k_idx.tolist(), " probs:", probs.tolist())
print("sampled id (same as _top_k_sample):", _top_k_sample(logits, k))

top-k indices: [1, 2]  probs: [0.7310585975646973, 0.2689414322376251]
sampled id (same as _top_k_sample): 2


## 5) Top-p (nucleus) sampling

1. Sort tokens by **descending probability** (after softmax on full vocab).
2. Take the **smallest prefix** whose cumulative probability exceeds `p` (with the implementation’s tie rule: keep token if `cumsum - prob <= p`).
3. Renormalize probabilities over that set; **multinomial** one draw.

If `p >= 1` or `p <= 0`, the code **falls back to greedy**.

Effect: the number of candidates **adapts** to the distribution (narrow when one token dominates, wider when mass is spread).

In [5]:
torch.manual_seed(1)
logits = torch.tensor([2.0, 1.5, 1.0, 0.0, -3.0])
probs = F.softmax(logits.float(), dim=-1)
sorted_probs, sorted_idx = torch.sort(probs, descending=True)
cumsum = torch.cumsum(sorted_probs, dim=-1)
mask = cumsum - sorted_probs <= 0.9
print("sorted probs:", sorted_probs.tolist())
print("nucleus mask (p=0.9):", mask.tolist())
print("sampled id:", _top_p_sample(logits, 0.9))

sorted probs: [0.472481906414032, 0.28657475113868713, 0.17381638288497925, 0.06394347548484802, 0.003183558117598295]
nucleus mask (p=0.9): [True, True, True, False, False]
sampled id: 0


## 6) Order of operations in `generate()`

For each step, the pipeline is:

1. Forward model → `logits` (last position).
2. If `temperature != 1` and `temperature > 0`: `logits = logits / temperature`.
3. `_apply_repetition_penalty(logits.clone(), ids, repetition_penalty)`.
4. Greedy / top-k / top-p on the **modified** logits.

So temperature is applied **before** repetition penalty; both affect the distribution used for sampling.

## 7) Repetition penalty

For each token id already present in the generated sequence:

- If `logits[id] > 0`: divide by `penalty` (makes positive logits smaller → less likely).
- If `logits[id] <= 0`: multiply by `penalty` (pushes more negative).

`penalty == 1.0` means **no change**. Values like `1.1`–`1.5` reduce boring loops.

In [6]:
logits = torch.tensor([2.0, 2.0, 0.0, 0.0])
ids_so_far = [0, 1]  # pretend tokens 0 and 1 already appeared
penalized = _apply_repetition_penalty(logits.clone(), ids_so_far, penalty=1.5)
print("before:", logits.tolist())
print("after :", penalized.tolist())
print("greedy next (penalized):", _greedy_sample(penalized))

before: [2.0, 2.0, 0.0, 0.0]
after : [1.3333333730697632, 1.3333333730697632, 0.0, 0.0]
greedy next (penalized): 0


## 8) Stopping: newline and `stop_sequence`

- **`stop_at_newline=True` (default):** after appending a token, if decoding that single token contains `"\n"`, generation stops.
- **`stop_sequence`:** after each step, full decode of `ids`; if `stop_sequence in text`, stop (useful for IMDB `[/REVIEW]` etc.).

CLI: `--no-stop-newline`, `--stop-sequence` in `scripts/generate.py`.

## 9) Mini end-to-end: **random** weights + `generate()`

This shows the **API** only. Random logits are often **extremely peaked**, so greedy and top-p can pick the **same** token forever (e.g. `"Hiiiii…"`). We add **`repetition_penalty > 1`** here to reduce that.

In [31]:
from nano_llm.model import build_model
from nano_llm.tokenizer import CharTokenizer

# Random init only — not loaded from a checkpoint (see §10 for trained weights).
tok = CharTokenizer()
model = build_model(
    vocab_size=tok.vocab_size,
    d_model=64,
    num_heads=4,
    num_layers=2,
    d_ff=128,
    max_len=128,
    dropout=0.0,
    weight_tie=True,
    position_encoding="sinusoidal",
    block_attn_residuals=False,
)
model.eval()

prompt = "Hi"
common = dict(
    model=model,
    tokenizer=tok,
    prompt=prompt,
    max_new_tokens=40,
    max_context=128,
    stop_at_newline=False,
    seed=42,
    repetition_penalty=1.2,
)
g = generate(**common, method="greedy")
t = generate(**common, method="top_p", top_p=0.92, temperature=1.05)
print("(Nonsense / repetition is expected: random weights)\n")
print("greedy:\n", repr(g[:120]))
print("top_p :\n", repr(t[:120]))
print("same string?", g == t, "← greedy is deterministic; top-p samples, so often False")

(Nonsense / repetition is expected: random weights)

greedy:
 'Hiiiiiii!!!!!uuuuu((((((     mmmmm$$$$$$cc'
top_p :
 'HiiiiiiiYYYYYYDDDDIIIIIIUUUUUU$$$$$$'
same string? False ← greedy is deterministic; top-p samples, so often False
